In [1]:
import pandas as pd
import numpy as np

# Read data
print("Reading results.csv...")
df = pd.read_csv('results.csv')

print(f"Number of rows: {len(df)}")
print(f"Data columns: {df.columns.tolist()}")

# Compute the deviation for each row
print("\nComputing deviation...")
ports = ['subgradient_port1', 'subgradient_port2', 'subgradient_port3']
df['deviation'] = df[ports].apply(lambda row: np.sum(np.abs(row - row.mean())), axis=1)

# Compute statistics grouped by x, y, z
print("\nComputing statistics...")
grouped = df.groupby(['x', 'y', 'z']).agg(
    total_cost_mean=('total_cost', 'mean'),
    total_cost_std=('total_cost', 'std'),
    total_cost_n=('total_cost', 'count'),
    deviation_mean=('deviation', 'mean'),
    deviation_std=('deviation', 'std')
).reset_index()

# Compute the standard error
grouped['total_cost_se'] = grouped['total_cost_std'] / np.sqrt(grouped['total_cost_n'])
grouped['deviation_se'] = grouped['deviation_std'] / np.sqrt(grouped['total_cost_n'])

# Keep only the needed columns
result = grouped[['x', 'y', 'z', 'total_cost_mean', 'total_cost_se',
                  'deviation_mean', 'deviation_se', 'total_cost_n']]

# Save results
output_file = 'statistics_by_xyz.csv'
result.to_csv(output_file, index=False)

# Show results
print('\nData statistics complete!')
print(f'Number of unique x,y,z combinations: {len(result)}')
print('\nFirst 10 rows of the result:')
print(result.head(10).to_string())
print(f'\nResults saved to: {output_file}')



Reading results.csv...
Number of rows: 72900
Data columns: ['x', 'y', 'z', 'plane', 'run', 'total_cost', 'subgradient_port1', 'subgradient_port2', 'subgradient_port3']

Computing deviation...

Computing statistics...

Data statistics complete!
Number of unique x,y,z combinations: 217

First 10 rows of the result:
     x     y     z  total_cost_mean  total_cost_se  deviation_mean  deviation_se  total_cost_n
0  0.0  0.00  0.00      5429.447819       6.453317     1418.955556      7.032387           900
1  0.0  0.00  0.25      5913.641591       7.924198      910.977778      8.161988           600
2  0.0  0.00  0.50      6328.973016       7.936002      614.126667      4.680350           600
3  0.0  0.00  0.75      6678.182096       7.939688      758.164444      8.656314           600
4  0.0  0.00  1.00      6977.142430       7.904701     1119.668889      9.348826           600
5  0.0  0.00  1.25      7226.599500       7.923597     1457.975556      9.323004           600
6  0.0  0.00  1.50  

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Read the CSV file
df = pd.read_csv('statistics_by_xyz.csv')

# Show basic information about the data
print("Data shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

Data shape: (217, 8)

Column names:
['x', 'y', 'z', 'total_cost_mean', 'total_cost_se', 'deviation_mean', 'deviation_se', 'total_cost_n']

First 5 rows:


,x,y,z,total_cost_mean,total_cost_se,deviation_mean,deviation_se,total_cost_n
0,0.0,0.0,0.00,5429.447819,6.453317,1418.955556,7.032387,900
1,0.0,0.0,0.25,5913.641591,7.924198,910.977778,8.161988,600
2,0.0,0.0,0.50,6328.973016,7.936002,614.126667,4.680350,600
3,0.0,0.0,0.75,6678.182096,7.939688,758.164444,8.656314,600
4,0.0,0.0,1.00,6977.142430,7.904701,1119.668889,9.348826,600


In [3]:
import scipy.stats as stats
overall_confidence=0.95
number_solution=len(df)

confidence_per_objective=1-(1-overall_confidence)/2
print(confidence_per_objective)

confidence_per_solution=1-confidence_per_objective**(number_solution)
print(confidence_per_solution)

z_score=np.abs(stats.norm.ppf((1-confidence_per_objective)/2))

0.975
0.9958884875792943


In [4]:
total_cost_mean=df['total_cost_mean'].tolist()
total_cost_se=df['total_cost_se'].tolist()
deviation_mean=df['deviation_mean'].tolist()
deviation_se=df['deviation_se'].tolist()
total_cost_mean=np.array(total_cost_mean)
total_cost_se=np.array(total_cost_se)
deviation_mean=np.array(deviation_mean)
deviation_se=np.array(deviation_se)
total_cost_lower=total_cost_mean-z_score*total_cost_se
total_cost_upper=total_cost_mean+z_score*total_cost_se
deviation_lower=deviation_mean-z_score*deviation_se
deviation_upper=deviation_mean+z_score*deviation_se
LB_list=np.array([-total_cost_upper,deviation_lower])
UB_list=np.array([-total_cost_lower,deviation_upper])
solution=np.array(df[['x','y','z']])

In [5]:
import gurobipy as gp
from gurobipy import GRB

LB=deviation_lower
UB=deviation_upper
Cst_matrix=np.zeros((3,1))
for solution_idx in range(3):
    model = gp.Model('Plausible Screening With Functional Info')
        
    # Create variables
    m = np.empty((number_solution), dtype=object)
    lip_cst=np.empty((3), dtype=object)

    for i in range(number_solution):
                name = f'm_{i}'
                m[i]= model.addVar(lb=LB[i], ub=UB[i], vtype=GRB.CONTINUOUS, name=name)

    for i in range(3):
                name = f'lip_cst_{i}'
                lip_cst[i]= model.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, vtype=GRB.CONTINUOUS, name=name)

    for i in range(number_solution):
        print(i)
        for j in range(i+1, number_solution):
            # Precompute the constant coefficients: |solution[i][l] - solution[j][l]|
            coeff = [abs(solution[i][l] - solution[j][l]) for l in range(3)]
            rhs_expr = gp.quicksum(coeff[l] * lip_cst[l] for l in range(3))

            model.addConstr(m[i] - m[j] <= rhs_expr, name=f"lip_pos_{i}_{j}")
            model.addConstr(m[j] - m[i] <= rhs_expr, name=f"lip_neg_{i}_{j}")

    model.setObjective(lip_cst[solution_idx], sense=GRB.MINIMIZE)

    model.optimize()
    inference_result=model.objVal
    Cst_matrix[solution_idx]=inference_result

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2678051
WLS license 2678051 - registered to C3.ai
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.5.0 25F84)

CPU model: Apple M4 Max


In [6]:
Cst_matrix

array([[2188.94226592],
       [1291.56311866],
       [2250.7515523 ]])

In [7]:
import gurobipy as gp
from gurobipy import GRB

Cst_array=np.zeros(3)

LB=LB_list[1]
UB=UB_list[1]
model = gp.Model('Plausible Screening With Functional Info')
    
# Create variables
m = np.empty((number_solution), dtype=object)

for i in range(number_solution):
            name = f'm_{i}'
            m[i]= model.addVar(lb=LB[i], ub=UB[i], vtype=GRB.CONTINUOUS, name=name)

lip_cst= model.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, vtype=GRB.CONTINUOUS, name=name)

for i in range(number_solution):
    print(i)
    for j in range(i+1, number_solution):
        # Precompute the constant coefficients: |solution[i][l] - solution[j][l]|
        dist_ij = np.linalg.norm(np.array(solution[i]) - np.array(solution[j]))
        rhs_expr = dist_ij * lip_cst

        model.addConstr(m[i] - m[j] <= rhs_expr, name=f"lip_pos_{i}_{j}")
        model.addConstr(m[j] - m[i] <= rhs_expr, name=f"lip_neg_{i}_{j}")

model.setObjective(lip_cst, sense=GRB.MINIMIZE)

model.optimize()
inference_result=model.objVal
print(inference_result)
Cst_lip=inference_result
print(Cst_lip)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.5.0 25F84)

CPU model: Apple M4 Max
Thread count: 16 physical cores, 16 logical processors, using up to 16 threads

WLS license 2678051 - registered to C3.ai
Optimize a 